# Module 3a: Ray Core - Tasks and Actors

**DSC 232R - Big Data Analysis Using Spark**

This notebook covers Ray Core fundamentals:
1. Ray initialization and architecture
2. Remote functions (Tasks)
3. Remote classes (Actors)
4. Object Store and memory management

**Prerequisites**: `pip install ray[default]`

## Key Takeaways

- **`@ray.remote`** transforms functions into distributed tasks
- **`.remote()`** submits work and returns immediately with an ObjectRef
- **`ray.get()`** blocks until results are ready
- **Actors** maintain state across method calls

---

## 1. Ray Initialization

In [ ]:
import ray
import time
import numpy as np

# Initialize Ray (local mode for this notebook)
# On a cluster, you'd use ray.init(address='auto')
if ray.is_initialized():
    ray.shutdown()

ray.init(
    num_cpus=4,  # Limit CPUs for notebook
    ignore_reinit_error=True,
    logging_level="WARNING"
)

print(f"Ray version: {ray.__version__}")
print(f"Available resources: {ray.available_resources()}")

### Understanding the Ray Dashboard

When Ray is running, a dashboard is available at `http://localhost:8265` showing:
- Cluster status and resources
- Running tasks and actors
- Memory usage
- Logs and errors

---

## 2. Ray Tasks

### Basic Task Definition

Use `@ray.remote` to make any function run on a worker:

In [ ]:
# Regular Python function
def square(x):
    return x * x

# Ray remote function (task)
@ray.remote
def square_remote(x):
    time.sleep(0.1)  # Simulate work
    return x * x

# Compare execution
print("Local execution:")
result_local = square(4)
print(f"  Result: {result_local}")

print("\nRemote execution:")
future = square_remote.remote(4)
print(f"  Future (ObjectRef): {future}")
result_remote = ray.get(future)
print(f"  Result: {result_remote}")

### Parallel Execution

The real power comes from running multiple tasks in parallel:

In [ ]:
# Sequential execution (slow)
def sequential_squares(numbers):
    results = []
    for n in numbers:
        time.sleep(0.1)
        results.append(n * n)
    return results

# Parallel execution with Ray (fast)
def parallel_squares(numbers):
    # Submit all tasks at once
    futures = [square_remote.remote(n) for n in numbers]
    # Wait for all results
    return ray.get(futures)

numbers = list(range(20))

# Time sequential
start = time.time()
seq_results = sequential_squares(numbers)
seq_time = time.time() - start

# Time parallel
start = time.time()
par_results = parallel_squares(numbers)
par_time = time.time() - start

print(f"Sequential time: {seq_time:.2f}s")
print(f"Parallel time:   {par_time:.2f}s")
print(f"Speedup:         {seq_time/par_time:.1f}x")
print(f"\nResults match: {seq_results == par_results}")

### Task Dependencies

Ray automatically handles dependencies between tasks:

In [ ]:
@ray.remote
def add(a, b):
    print(f"  Computing {a} + {b}")
    time.sleep(0.2)
    return a + b

@ray.remote
def multiply(a, b):
    print(f"  Computing {a} * {b}")
    time.sleep(0.2)
    return a * b

# Create a task graph
print("Building task graph...")
x = add.remote(1, 2)         # Task 1
y = add.remote(3, 4)         # Task 2 (parallel with Task 1)
z = multiply.remote(x, y)    # Task 3 (waits for x and y)

print("\nExecuting (tasks run in parallel where possible):")
start = time.time()
result = ray.get(z)
elapsed = time.time() - start

print(f"\nResult: (1+2) * (3+4) = {result}")
print(f"Time: {elapsed:.2f}s (two 0.2s tasks in parallel + one 0.2s task)")

### Resource Requirements

Specify CPU, GPU, and memory requirements per task:

In [ ]:
# Task requiring specific resources
@ray.remote(num_cpus=2)  # Requires 2 CPUs
def cpu_intensive_task(data):
    return np.sum(data ** 2)

# Override resources at call time
data = np.random.rand(1000)

# Using decorator defaults (2 CPUs)
future1 = cpu_intensive_task.remote(data)

# Override to use 1 CPU
future2 = cpu_intensive_task.options(num_cpus=1).remote(data)

results = ray.get([future1, future2])
print(f"Results: {results[0]:.2f}, {results[1]:.2f}")

---

## 3. Ray Actors

Actors are stateful - they maintain state across method calls:

In [ ]:
@ray.remote
class Counter:
    """A simple counter that maintains state."""
    
    def __init__(self, initial_value=0):
        self.value = initial_value
        print(f"Counter initialized with value {initial_value}")
    
    def increment(self, amount=1):
        self.value += amount
        return self.value
    
    def get_value(self):
        return self.value

# Create actor instance
counter = Counter.remote(10)

# Call methods
print(f"Initial: {ray.get(counter.get_value.remote())}")
print(f"After +1: {ray.get(counter.increment.remote())}")
print(f"After +5: {ray.get(counter.increment.remote(5))}")
print(f"Current: {ray.get(counter.get_value.remote())}")

### Actor for ML Model Inference

A practical example - serving an ML model:

In [ ]:
from sklearn.linear_model import LinearRegression

@ray.remote
class ModelServer:
    """Actor that serves ML model predictions."""
    
    def __init__(self):
        # Train a simple model
        X = np.random.rand(100, 5)
        y = X.sum(axis=1) + np.random.randn(100) * 0.1
        
        self.model = LinearRegression()
        self.model.fit(X, y)
        self.prediction_count = 0
        print("ModelServer initialized and trained")
    
    def predict(self, features):
        """Make prediction and track count."""
        self.prediction_count += 1
        features = np.array(features).reshape(1, -1)
        return float(self.model.predict(features)[0])
    
    def get_stats(self):
        return {
            "prediction_count": self.prediction_count,
            "coefficients": self.model.coef_.tolist()
        }

# Create model server
server = ModelServer.remote()

# Make predictions
test_features = [[0.5, 0.3, 0.8, 0.2, 0.1],
                 [0.1, 0.9, 0.4, 0.6, 0.7],
                 [0.8, 0.2, 0.5, 0.3, 0.9]]

for features in test_features:
    pred = ray.get(server.predict.remote(features))
    print(f"Features: {features} -> Prediction: {pred:.3f}")

stats = ray.get(server.get_stats.remote())
print(f"\nServer stats: {stats}")

### Multiple Actors

You can create multiple actor instances for parallel stateful processing:

In [ ]:
@ray.remote
class DataProcessor:
    """Actor that processes data and accumulates results."""
    
    def __init__(self, processor_id):
        self.id = processor_id
        self.total = 0
        self.count = 0
    
    def process(self, data):
        result = sum(data)
        self.total += result
        self.count += len(data)
        return result
    
    def get_stats(self):
        return {
            "id": self.id,
            "total": self.total,
            "count": self.count,
            "mean": self.total / self.count if self.count > 0 else 0
        }

# Create pool of processors
num_processors = 3
processors = [DataProcessor.remote(i) for i in range(num_processors)]

# Distribute work across processors
data_batches = [list(range(i*10, (i+1)*10)) for i in range(9)]

futures = []
for i, batch in enumerate(data_batches):
    processor = processors[i % num_processors]  # Round-robin
    futures.append(processor.process.remote(batch))

# Wait for all processing
results = ray.get(futures)
print(f"Batch results: {results}")

# Get stats from each processor
print("\nProcessor stats:")
for processor in processors:
    stats = ray.get(processor.get_stats.remote())
    print(f"  Processor {stats['id']}: processed {stats['count']} items, total={stats['total']}")

---

## 4. Object Store

Ray's object store enables efficient data sharing:

In [ ]:
# Create a large array
large_array = np.random.rand(1_000_000)
print(f"Array size: {large_array.nbytes / 1024 / 1024:.2f} MB")

# Put in object store (once)
array_ref = ray.put(large_array)
print(f"Object reference: {array_ref}")

In [ ]:
@ray.remote
def compute_stats(data_ref):
    """Compute statistics on shared data."""
    # ray.get() inside task retrieves from local object store (zero-copy)
    data = ray.get(data_ref)
    return {
        "mean": float(np.mean(data)),
        "std": float(np.std(data)),
        "min": float(np.min(data)),
        "max": float(np.max(data))
    }

@ray.remote
def compute_percentiles(data_ref, percentiles):
    """Compute percentiles on shared data."""
    data = ray.get(data_ref)
    return {f"p{p}": float(np.percentile(data, p)) for p in percentiles}

# Multiple tasks share the same data (no copying)
stats_future = compute_stats.remote(array_ref)
percentiles_future = compute_percentiles.remote(array_ref, [25, 50, 75])

stats, percentiles = ray.get([stats_future, percentiles_future])
print("Statistics:", stats)
print("Percentiles:", percentiles)

### Memory Efficiency Comparison

In [ ]:
@ray.remote
def process_with_copy(data):
    """This copies data to each task (inefficient)."""
    return np.mean(data)

@ray.remote
def process_with_ref(data_ref):
    """This uses shared reference (efficient)."""
    data = ray.get(data_ref)
    return np.mean(data)

# Create data
data = np.random.rand(100_000)
data_ref = ray.put(data)

num_tasks = 10

# Method 1: Pass data directly (copies each time)
start = time.time()
futures_copy = [process_with_copy.remote(data) for _ in range(num_tasks)]
results_copy = ray.get(futures_copy)
time_copy = time.time() - start

# Method 2: Pass reference (zero-copy)
start = time.time()
futures_ref = [process_with_ref.remote(data_ref) for _ in range(num_tasks)]
results_ref = ray.get(futures_ref)
time_ref = time.time() - start

print(f"With data copy:  {time_copy:.3f}s")
print(f"With object ref: {time_ref:.3f}s")
print(f"Speedup: {time_copy/time_ref:.1f}x")
print(f"\nNote: Object store method is more memory efficient for large data")

---

## 5. Exercise: Parallel Monte Carlo Pi Estimation

Estimate Pi using the Monte Carlo method, parallelized with Ray:

In [ ]:
# Exercise: Complete this function

@ray.remote
def estimate_pi_chunk(num_samples):
    """
    Estimate pi using Monte Carlo method.
    
    Algorithm:
    1. Generate random (x, y) points in unit square [0,1] x [0,1]
    2. Count how many fall inside quarter circle (x^2 + y^2 <= 1)
    3. Pi/4 = (points in circle) / (total points)
    
    Returns: count of points inside the quarter circle
    """
    # Your code here
    # Hint: Use np.random.uniform(0, 1, size=(num_samples, 2))
    pass

def parallel_pi_estimation(total_samples, num_chunks):
    """
    Estimate pi by distributing work across multiple tasks.
    
    Returns: estimated value of pi
    """
    samples_per_chunk = total_samples // num_chunks
    
    # Your code here
    # 1. Launch num_chunks tasks
    # 2. Collect results with ray.get()
    # 3. Calculate pi from total inside count
    pass

# Test your implementation
# pi_estimate = parallel_pi_estimation(10_000_000, 10)
# print(f"Estimated Pi: {pi_estimate:.6f}")
# print(f"Actual Pi:    {np.pi:.6f}")
# print(f"Error:        {abs(pi_estimate - np.pi):.6f}")

In [ ]:
# Solution

@ray.remote
def estimate_pi_chunk_solution(num_samples):
    """Estimate pi using Monte Carlo - one chunk."""
    points = np.random.uniform(0, 1, size=(num_samples, 2))
    inside_circle = np.sum(points[:, 0]**2 + points[:, 1]**2 <= 1)
    return inside_circle

def parallel_pi_estimation_solution(total_samples, num_chunks):
    """Estimate pi using parallel Monte Carlo."""
    samples_per_chunk = total_samples // num_chunks
    
    # Launch all tasks
    futures = [estimate_pi_chunk_solution.remote(samples_per_chunk) 
               for _ in range(num_chunks)]
    
    # Collect results
    inside_counts = ray.get(futures)
    total_inside = sum(inside_counts)
    
    # Pi/4 = inside/total, so Pi = 4 * inside/total
    return 4 * total_inside / (samples_per_chunk * num_chunks)

# Run solution
start = time.time()
pi_estimate = parallel_pi_estimation_solution(10_000_000, 10)
elapsed = time.time() - start

print(f"Estimated Pi: {pi_estimate:.6f}")
print(f"Actual Pi:    {np.pi:.6f}")
print(f"Error:        {abs(pi_estimate - np.pi):.6f}")
print(f"Time:         {elapsed:.2f}s")

---

## 6. Best Practices

### When to Use Tasks vs Actors

| Use Case | Tasks | Actors |
|----------|-------|--------|
| Stateless computation | Yes | No |
| Need to maintain state | No | Yes |
| Embarrassingly parallel | Yes | No |
| Sequential method calls | No | Yes |
| ML model inference | Either | Better |
| Data aggregation | Yes | Sometimes |

### Memory Tips

1. Use `ray.put()` for large data shared by multiple tasks
2. Avoid returning large results from tasks
3. Use `ray.wait()` for streaming results instead of `ray.get()` on all at once

In [ ]:
# Using ray.wait() for streaming results

@ray.remote
def slow_task(i):
    time.sleep(np.random.uniform(0.1, 0.5))
    return f"Task {i} complete"

# Launch tasks
futures = [slow_task.remote(i) for i in range(5)]

# Process results as they complete (instead of waiting for all)
print("Processing results as they complete:")
while futures:
    done, futures = ray.wait(futures, num_returns=1)
    result = ray.get(done[0])
    print(f"  {result}")

---

## Summary

### Ray Core Concepts

1. **Tasks** (`@ray.remote` on functions)
   - Stateless parallel execution
   - `.remote()` returns ObjectRef immediately
   - `ray.get()` retrieves results

2. **Actors** (`@ray.remote` on classes)
   - Stateful distributed objects
   - Methods execute sequentially on one actor
   - Great for ML model serving

3. **Object Store**
   - `ray.put()` stores data
   - Zero-copy sharing between tasks
   - Use for large shared data

### Next Steps

- `03b_ray_data.ipynb`: Distributed datasets
- `03c_ray_train.ipynb`: Distributed training

In [ ]:
# Cleanup
ray.shutdown()
print("Ray shutdown complete.")